# Week 5 AutoGen Core Exercise: Support Triage Agent System

This notebook is fully runnable and uses Week 5 AutoGen Core concepts:
- custom message protocol (dataclasses)
- multiple `RoutedAgent` workers
- inter-agent communication via `send_message`
- `SingleThreadedAgentRuntime` registration/start/stop


## 1) Imports and Environment


In [ ]:
from dataclasses import dataclass
from typing import Literal
import os

from dotenv import load_dotenv

from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient

load_dotenv(override=True)

if not os.getenv('OPENAI_API_KEY'):
    raise ValueError('OPENAI_API_KEY is missing in .env')


## 2) Define Message Protocol

These dataclasses are the typed messages passed between agents.


In [ ]:
@dataclass
class SupportRequest:
    customer_name: str
    issue_text: str


@dataclass
class ClassifiedRequest:
    customer_name: str
    issue_text: str
    severity: Literal['low', 'medium', 'high']
    topic: str


@dataclass
class DraftReply:
    customer_name: str
    severity: str
    topic: str
    reply_text: str


@dataclass
class ReviewResult:
    approved: bool
    feedback: str


@dataclass
class FinalResponse:
    customer_name: str
    severity: str
    topic: str
    final_reply: str
    reviewer_feedback: str


## 3) Build Routed Agents

Flow:
1. `CoordinatorAgent` receives `SupportRequest`.
2. Coordinator asks `ClassifierAgent` to classify severity/topic.
3. Coordinator asks `ResponderAgent` to draft response.
4. Coordinator asks `ReviewerAgent` to approve or request one revision.


In [ ]:
class ClassifierAgent(RoutedAgent):
    def __init__(self):
        super().__init__('ClassifierAgent')
        self._delegate = AssistantAgent(
            name='classifier_delegate',
            model_client=OpenAIChatCompletionClient(model='gpt-4o-mini')
        )

    @message_handler
    async def classify(self, message: SupportRequest, ctx: MessageContext) -> ClassifiedRequest:
        prompt = (
            'Classify the support issue.\n'
            f'Customer: {message.customer_name}\n'
            f'Issue: {message.issue_text}\n\n'
            'Return exactly two lines:\n'
            'severity: one of low, medium, high\n'
            'topic: short topic label'
        )
        response = await self._delegate.on_messages([TextMessage(content=prompt, source='user')], ctx.cancellation_token)
        text = (response.chat_message.content or '').strip().lower()

        severity = 'medium'
        topic = 'general_support'

        for line in text.splitlines():
            if line.startswith('severity:'):
                val = line.split(':', 1)[1].strip()
                if val in {'low', 'medium', 'high'}:
                    severity = val
            if line.startswith('topic:'):
                topic = line.split(':', 1)[1].strip().replace(' ', '_') or topic

        return ClassifiedRequest(
            customer_name=message.customer_name,
            issue_text=message.issue_text,
            severity=severity,
            topic=topic,
        )


class ResponderAgent(RoutedAgent):
    def __init__(self):
        super().__init__('ResponderAgent')
        self._delegate = AssistantAgent(
            name='responder_delegate',
            model_client=OpenAIChatCompletionClient(model='gpt-4o-mini')
        )

    @message_handler
    async def draft(self, message: ClassifiedRequest, ctx: MessageContext) -> DraftReply:
        prompt = (
            'Write a professional support reply to the customer.\n'
            f'Customer: {message.customer_name}\n'
            f'Severity: {message.severity}\n'
            f'Topic: {message.topic}\n'
            f'Issue: {message.issue_text}\n\n'
            'Keep it concise, empathetic, and actionable.'
        )
        response = await self._delegate.on_messages([TextMessage(content=prompt, source='user')], ctx.cancellation_token)
        reply = (response.chat_message.content or '').strip()

        return DraftReply(
            customer_name=message.customer_name,
            severity=message.severity,
            topic=message.topic,
            reply_text=reply,
        )


class ReviewerAgent(RoutedAgent):
    def __init__(self):
        super().__init__('ReviewerAgent')
        self._delegate = AssistantAgent(
            name='reviewer_delegate',
            model_client=OpenAIChatCompletionClient(model='gpt-4o-mini')
        )

    @message_handler
    async def review(self, message: DraftReply, ctx: MessageContext) -> ReviewResult:
        prompt = (
            'Review this support draft.\n'
            f'Severity: {message.severity}\n'
            f'Topic: {message.topic}\n'
            f'Draft:\n{message.reply_text}\n\n'
            'Return exactly two lines:\n'
            'approved: yes or no\n'
            'feedback: short feedback'
        )
        response = await self._delegate.on_messages([TextMessage(content=prompt, source='user')], ctx.cancellation_token)
        text = (response.chat_message.content or '').strip().lower()

        approved = False
        feedback = 'Improve clarity and action steps.'

        for line in text.splitlines():
            if line.startswith('approved:'):
                val = line.split(':', 1)[1].strip()
                approved = val.startswith('y')
            if line.startswith('feedback:'):
                feedback = line.split(':', 1)[1].strip() or feedback

        return ReviewResult(approved=approved, feedback=feedback)


class CoordinatorAgent(RoutedAgent):
    def __init__(self):
        super().__init__('CoordinatorAgent')

    @message_handler
    async def orchestrate(self, message: SupportRequest, ctx: MessageContext) -> FinalResponse:
        classifier_id = AgentId('classifier', 'default')
        responder_id = AgentId('responder', 'default')
        reviewer_id = AgentId('reviewer', 'default')

        classified = await self.send_message(message, classifier_id)
        draft = await self.send_message(classified, responder_id)
        review = await self.send_message(draft, reviewer_id)

        if not review.approved:
            revised_input = ClassifiedRequest(
                customer_name=classified.customer_name,
                issue_text=(
                    classified.issue_text
                    + f"\n\nReviewer feedback to incorporate: {review.feedback}"
                ),
                severity=classified.severity,
                topic=classified.topic,
            )
            draft = await self.send_message(revised_input, responder_id)
            review = await self.send_message(draft, reviewer_id)

        return FinalResponse(
            customer_name=draft.customer_name,
            severity=draft.severity,
            topic=draft.topic,
            final_reply=draft.reply_text,
            reviewer_feedback=review.feedback,
        )


## 4) Register Agents in Runtime


In [ ]:
runtime = SingleThreadedAgentRuntime()

await ClassifierAgent.register(runtime, 'classifier', lambda: ClassifierAgent())
await ResponderAgent.register(runtime, 'responder', lambda: ResponderAgent())
await ReviewerAgent.register(runtime, 'reviewer', lambda: ReviewerAgent())
await CoordinatorAgent.register(runtime, 'coordinator', lambda: CoordinatorAgent())

runtime.start()
print('Runtime started.')


## 5) Send a Request to Coordinator


In [ ]:
coordinator_id = AgentId('coordinator', 'default')

request = SupportRequest(
    customer_name='Amina',
    issue_text='I was billed twice for my monthly subscription and I need this fixed quickly.'
)

result = await runtime.send_message(request, coordinator_id)

print('Customer:', result.customer_name)
print('Severity:', result.severity)
print('Topic:', result.topic)
print('Reviewer feedback:', result.reviewer_feedback)
print('\nFinal reply:\n')
print(result.final_reply)


## 6) Stop Runtime


In [ ]:
await runtime.stop()
await runtime.close()
print('Runtime stopped and closed.')
